In [218]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

In [219]:
link_hist = '/Users/artemvorobchenko/Desktop/traiding_data/noHistory (2)/normalized-2026-01-02.csv'
link_profit = '/Users/artemvorobchenko/Desktop/traiding_data/noHistory (2)/profitable_trades_main.csv'

In [220]:
cols_noHistory = pd.read_csv(link_hist, nrows=1).columns

num_cols = [col for col in all_cols if 'symbol' not in col]

In [221]:
noHistory = pd.read_csv(link_hist, usecols=num_cols)
profit = pd.read_csv(link_profit)

# isnull stats

In [222]:
noHistory.isna().sum().sum() / noHistory.size * 100

np.float64(19.310344827586206)

In [223]:
noHistory.isnull().mean() * 100

uPrice                0.000000
volatility            0.000000
call_-13_ask         94.978987
call_-13_askSize     94.978987
call_-13_bid         94.978987
                       ...    
put_13_rho           94.934749
put_13_theta         94.934749
put_13_vega          94.934749
put_13_volatility    94.934749
put_13_realPrice     94.934749
Length: 870, dtype: float64

In [224]:
noHistory_clear = noHistory.dropna(thresh=(len(noHistory) * 0.5), axis=1)

#noHistory_clear.columns[noHistory_clear.isnull().any()].tolist()

In [225]:
noHistory_clear.isna().sum().sum() / noHistory_clear.size * 100

np.float64(4.532063412842001)

In [226]:
noHistory_clear = noHistory_clear.copy()

noHistory_clear.ffill(inplace=True)
noHistory_clear.bfill(inplace=True)

In [227]:
print(f'NaN: {noHistory_clear.isna().sum().sum() / noHistory_clear.size * 100}%')

NaN: 0.0%


# Train/valid/test

* 70% train
* 15% valid
* 15% test

In [228]:
profit = profit.sort_values("buy_row_idx").reset_index(drop=True)

### поиск уникальных индексов покупки (делается для того, чтобы при разбиении на train/valid/test одинаковые индексы не попали в разные выборки, и не произошла утечка)

### после оставим только индексы >= 15, чтобы мы могли учитывать полную историю в тензоре

In [229]:
unique_buy_idx = profit["buy_row_idx"].unique()
unique_buy_idx = unique_buy_idx[unique_buy_idx >= 15]
unique_buy_idx = sorted(unique_buy_idx)

# еще важный момент, расстояние между train/valid/test должны быть минимум в 15 объектов, иначе получится, что например в valid попадет кусок данных из train.

* для этого нам просто нужно прибавлять к концам интревалов наш window_size

In [230]:
n_states = len(unique_buy_idx)
window_size = 16

train_end_idx = int(n_states * 0.70)
valid_end_idx = int(n_states * 0.85)

train_states = unique_buy_idx[:train_end_idx]
valid_states = unique_buy_idx[train_end_idx + window_size : valid_end_idx]
test_states  = unique_buy_idx[valid_end_idx + window_size :]

# ___________________________________________________

In [231]:
print("Train states:", len(train_states))
print("Valid states:", len(valid_states))
print("Test states:", len(test_states))

Train states: 2702
Valid states: 563
Test states: 563


In [232]:
train_states_set = set(train_states)
valid_states_set = set(valid_states)
test_states_set  = set(test_states)

profit_train = profit[profit["buy_row_idx"].isin(train_states_set)].copy()
profit_valid = profit[profit["buy_row_idx"].isin(valid_states_set)].copy()
profit_test  = profit[profit["buy_row_idx"].isin(test_states_set)].copy()

In [233]:
print(profit["type"].value_counts(dropna=False))

type
Temporal_CALL_1    438898
Temporal_PUT_1     302333
Name: count, dtype: int64


In [234]:
profit.groupby("type")["buy_row_idx"].agg(["min", "max", "count"])

,min,max,count
type,,,
Temporal_CALL_1,10,4419,438898
Temporal_PUT_1,10,4418,302333


In [235]:
print("TRAIN")
print(profit_train["type"].value_counts(normalize=True))

print("\nVALID")
print(profit_valid["type"].value_counts(normalize=True))

print("\nTEST")
print(profit_test["type"].value_counts(normalize=True))

TRAIN
type
Temporal_CALL_1    0.589544
Temporal_PUT_1     0.410456
Name: proportion, dtype: float64

VALID
type
Temporal_CALL_1    0.632496
Temporal_PUT_1     0.367504
Name: proportion, dtype: float64

TEST
type
Temporal_CALL_1    0.778309
Temporal_PUT_1     0.221691
Name: proportion, dtype: float64


### не забыть в будущем чек кластера профит рэтио по put/call

# ____________________________________________________

# Функция окон, берем 16 строк рынка

* тензор [samples \ timesteps \ features]

In [236]:
def build_tensor(shop_df, state_indices, window_size=16):
    samples = []
    used_states = []

    for t in state_indices:
        if t >= window_size - 1:
            seq = shop_df.iloc[t - window_size + 1 : t + 1].values
            samples.append(seq)
            used_states.append(t)

    X = np.stack(samples)

    return X, np.array(used_states)

In [237]:
X_train, train_states_used = build_tensor(noHistory_clear, train_states)
X_valid, valid_states_used = build_tensor(noHistory_clear, valid_states)
X_test, test_states_used = build_tensor(noHistory_clear, test_states)

### и того у нас есть
* 3860 уникальных индексов = 2702 + 579 + 579

на каждый индекс у нас приходится по временому срезу из 16 объектов

* 2702*16 = 43232
* 579*16  = 9264
* 579*16  = 9264

СУММА ВСЕГО ДОБРА 61760 уникальных строк для обучения энкодера